# ABSA Experiment
Experiment to make extract keywords from the task 1 and passing the keywords result to task 3

Author: Raihan Muhammad Daffa<br/>
NIM: 18223140

## Import Modules

In [1]:
import os
import sys

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
    
import re
import json
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.utils.data import Dataset
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

## GPU Utilization Check

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## Helper

In [3]:
def build_prompt(review_text):
    return(
        "### Instruction: Extract the key aspects and their sentiments from the following game review"
        "You MUST format the output strictly as Aspects: "
        "[category:aspect:sentiment, ...]\n\n"
        f"### Input: Review: \"{review_text}\"\n\n"
        "### Response:"
    )
    
def parse_aspect_tuples(text):
    matches = re.findall(r"([A-Za-z0-9_/]+):([^:\],]+):([A-Za-z]+)", text)
    return set((cat.strip().lower(), asp.strip().lower(), sent.strip().lower()) for cat, asp, sent in matches)

def evaluate_model(model, tokenizer, test_data, experiment_name):
    model.eval()
    exact_matches = 0
    total_p, total_r, total_f1 = 0.0, 0.0, 0.0
    n = len(test_data)
    
    for item in test_data:
        inputs = tokenizer(item["input_text"], return_tensors="pt", max_length=512, truncation=True).to(device)
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                no_repeat_ngram_size=3,
                repetition_penalty=1.5,
                early_stopping=True
            )
            
        pred_text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        gold_text = item["target_text"]
        
        if pred_text.strip().lower() == gold_text.strip().lower():
            exact_matches += 1

        pred_triplets = parse_aspect_tuples(pred_text)
        gold_triplets = parse_aspect_tuples(gold_text)
        intersection = len(pred_triplets & gold_triplets)
        
        if pred_triplets:
            prec = intersection / len(pred_triplets)
        else:
            prec = 0.0
            
        if gold_triplets:
            rec = intersection / len(gold_triplets)
        else:  
            rec = 0.0
        
        if prec + rec > 0:
            f1 = 2 * (prec * rec) / (prec + rec)
        else:
            f1 = 0.0
        
        total_p += prec
        total_r += rec
        total_f1 += f1
        
    results = {
        "Experiment": experiment_name,
        "Exact Match (%)": round((exact_matches / n) * 100, 2),
        "Precision (%)": round((total_p / n) * 100, 2),
        "Recall (%)": round((total_r / n) * 100, 2),
        "F1-Score (%)": round((total_f1 / n) * 100, 2)
    }
    return results

all_expe_results = []

In [4]:
class ABSADataset(Dataset):
    def __init__(self, data_pairs, tokenizer, max_input_len=512, max_target_len=128):
        self.data_pairs = data_pairs
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
        
    def __len__(self):
        return len(self.data_pairs)
    
    def __getitem__(self, idx):
        item = self.data_pairs[idx]
        model_inputs = self.tokenizer(item["input_text"], max_length=self.max_input_len, truncation=True)
        labels = self.tokenizer(text_target=item["target_text"], max_length=self.max_target_len, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

## Load the dataset

In [5]:
INPUT_FILE = os.path.join(BASE_DIR, "data", "preprocessed", "dataset_labeled.jsonl")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

data_pairs = []
for game_id, game_data in raw_data.items():
    reviews = game_data.get("reviews", [])
    target = game_data.get("target_label", "")
    combined = " ".join(reviews)
    if combined and target:
        data_pairs.append({
            "game": game_data.get("app_name", f"Game {game_id}"),
            "input_text": build_prompt(combined),
            "target_text": target
        })
        

## Split the data

In [6]:
split_idx = int(0.8 * len(data_pairs))
train_data = data_pairs[:split_idx]
test_data = data_pairs[split_idx:]
print(f"Total samples: {len(data_pairs)} | Train: {len(train_data)} | Test: {len(test_data)}")

Total samples: 300 | Train: 240 | Test: 60


## Experiment 1 (Zero-Base Shot)

In [7]:
base_model_name = "facebook/bart-base"
bart_tokenizer = BartTokenizer.from_pretrained(base_model_name)
bart_zero_shot_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [8]:
res_zero_shot = evaluate_model(bart_zero_shot_model, bart_tokenizer, test_data, "Zero-Shot (Raw BART)")
all_expe_results.append(res_zero_shot)
print(pd.DataFrame([res_zero_shot]))

             Experiment  Exact Match (%)  Precision (%)  Recall (%)  \
0  Zero-Shot (Raw BART)              0.0            0.0         0.0   

   F1-Score (%)  
0           0.0  


In [9]:
del bart_zero_shot_model
torch.cuda.empty_cache()

## Experiment 2 (Task-Adapted BART)

In [10]:
bart_task_adapted_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)
bart_train_dataset = ABSADataset(train_data, bart_tokenizer)
bart_data_collator = DataCollatorForSeq2Seq(tokenizer=bart_tokenizer,model=bart_task_adapted_model)

bart_task_adapted_training_arguments = Seq2SeqTrainingArguments(
    output_dir="models/bart_3epochs",
    num_train_epochs=3,
    per_device_train_batch_size=2, 
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

bart_task_adapted_trainer = Seq2SeqTrainer(
    model=bart_task_adapted_model,
    args=bart_task_adapted_training_arguments,
    train_dataset=bart_train_dataset,
    data_collator=bart_data_collator
)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [11]:
bart_task_adapted_trainer.train()

Step,Training Loss
10,6.720576
20,4.410601
30,3.905937
40,3.286667
50,3.300840
60,3.324168
70,3.083700
80,2.967423
90,2.904738
100,2.767807


TrainOutput(global_step=180, training_loss=3.2081781281365287, metrics={'train_runtime': 54.2161, 'train_samples_per_second': 13.28, 'train_steps_per_second': 3.32, 'total_flos': 219505124966400.0, 'train_loss': 3.2081781281365287, 'epoch': 3.0})

In [12]:
bart_task_adapted_evaluation = evaluate_model(bart_task_adapted_model, bart_tokenizer, test_data, "Task-Adapted (BART - 3 Epochs)")

In [13]:
all_expe_results.append(bart_task_adapted_evaluation)
print(pd.DataFrame([bart_task_adapted_evaluation]))

                       Experiment  Exact Match (%)  Precision (%)  Recall (%)  \
0  Task-Adapted (BART - 3 Epochs)              0.0          12.92         1.4   

   F1-Score (%)  
0           2.5  


## Experiments 3

In [14]:
bart_hyperparametered_task_adapted_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)

bart_hyperparametered_task_adapted_training_arguments = Seq2SeqTrainingArguments(
    output_dir="models/bart_3epochs",
    num_train_epochs=4,
    per_device_train_batch_size=2, 
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    predict_with_generate=True,
    fp16=True,
    
    report_to="none"
)

bart_hyperparametered_task_adapted_trainer = Seq2SeqTrainer(
    model=bart_hyperparametered_task_adapted_model,
    args=bart_hyperparametered_task_adapted_training_arguments,
    train_dataset=bart_train_dataset,
    data_collator=bart_data_collator
)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [15]:
bart_hyperparametered_task_adapted_trainer.train()

Step,Training Loss
10,6.720884
20,4.407159
30,3.903116
40,3.280658
50,3.293501
60,3.320203
70,3.051344
80,2.948181
90,2.880746
100,2.755567


TrainOutput(global_step=240, training_loss=2.9632592678070067, metrics={'train_runtime': 300.3977, 'train_samples_per_second': 3.196, 'train_steps_per_second': 0.799, 'total_flos': 292673499955200.0, 'train_loss': 2.9632592678070067, 'epoch': 4.0})

In [16]:
bart_hyperparametered_task_adapted_evaluation = evaluate_model(bart_hyperparametered_task_adapted_model, bart_tokenizer, test_data, "Hyperparametered Task-Adapted (BART - 5 Epochs)")

In [17]:
all_expe_results.append(bart_hyperparametered_task_adapted_evaluation)
print(pd.DataFrame([bart_hyperparametered_task_adapted_evaluation]))

                                        Experiment  Exact Match (%)  \
0  Hyperparametered Task-Adapted (BART - 5 Epochs)              0.0   

   Precision (%)  Recall (%)  F1-Score (%)  
0          11.06        1.52          2.66  


## Evaluation

In [18]:
summary_df = pd.DataFrame(all_expe_results)
display(summary_df)

,Experiment,Exact Match (%),Precision (%),Recall (%),F1-Score (%)
0,Zero-Shot (Raw BART),0.0,0.00,0.00,0.00
1,Task-Adapted (BART - 3 Epochs),0.0,12.92,1.40,2.50
2,Hyperparametered Task-Adapted (BART - 5 Epochs),0.0,11.06,1.52,2.66


## Error Analysis

In [19]:
best_model = bart_task_adapted_model
best_model.eval()

for sample in test_data[:3]:
    inputs = bart_tokenizer(sample["input_text"], return_tensors="pt", max_length=512, truncation=True).to(device)
    
    with torch.no_grad():
        out_ids = best_model.generate(**inputs, max_new_tokens=128, num_beams=4)
    
    pred = bart_tokenizer.decode(out_ids[0], skip_special_tokens=True)
    print(f"\n Game: {sample['game']}")
    print(f"Ground Truth: {sample['target_text']}")
    print(f"Prediction: {pred}")



 Game: Dead Space
Ground Truth: Aspects: [Gameplay:Shoot:positive, Gameplay:move:positive, UI/UX:guide:positive, Gameplay:Play Dead Space:negative, Gameplay:Turn:negative, Audio:Surround sound:negative, Gameplay:Kill:negative, Gameplay:Moves:negative, Gameplay:Move:negative, Gameplay:Worth:positive, Gameplay:grab:positive, Gameplay:scariest thing:neutral, Gameplay:logo:neutral, Gameplay:scariest:neutral, Gameplay:Ch-check:positive, Gameplay:GARBLE:positive, Gameplay:MOTHER OF GOD:positive, Gameplay:scariest thing:negative, Gameplay:thing:negative, Gameplay:Dead Space Top-notch:negative, Gameplay:Space Top-notch atmosphere:negative, Gameplay:Event Horizon:negative, Gameplay:unofficial Alien game:positive, Gameplay:unofficial Alien:positive, Gameplay:Dead Space:positive, Gameplay:Scary Shooter:positive, Gameplay:Horror Third-Person Shooter:positive, Gameplay:nice Horror Third-Person:positive]
Prediction: Aspects: [Gameplay:Shoot everything that move:positive, Gameplay:Shot everything:po

## Export the JSON File

In [30]:
def parse_predicted_aspects(pred_text):
    cleaned = re.sub(r"Aspects\s*:\s*\[?", "", pred_text, flags=re.IGNORECASE)
    cleaned = cleaned.replace("]", "")
    
    matches = re.findall(r"([A-Za-z0-9_/]+):([^:\],]+):([A-Za-z]+)", pred_text)
    aspects_list = []
    for cat, asp, sent in matches:
        aspects_list.append({
            "category": cat.strip(),
            "aspect": asp.strip(),
            "sentiment": sent.strip().capitalize()
        })
    return aspects_list

In [31]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    games_data = json.load(f)
    
pred_output = {}
best_model.eval()

for game_id, data in games_data.items():
    reviews = data.get("reviews", [])
    combined_reviews = " ".join(reviews)
    
    prompt = build_prompt(combined_reviews)
    inputs = bart_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    with torch.no_grad():
        out_ids = best_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            early_stopping=True
        )
    pred_text = bart_tokenizer.decode(out_ids[0], skip_special_tokens=True)
    
    pred_output[game_id] = {
        "app_name": data.get("app_name", ""),
        "reviews": reviews,
        "aspects": parse_predicted_aspects(pred_text),
        "raw_prediction": pred_text
    }



In [29]:
pred_output_file = os.path.join(BASE_DIR, "data", "modeled" , "data_predicted.jsonl")
os.makedirs(os.path.dirname(pred_output_file), exist_ok=True)
with open(pred_output_file, "w", encoding="utf-8") as f:
    json.dump(pred_output, f, indent=4, ensure_ascii=False)